### **🪖 Demo Deteksi Helm (Notebook)**

Notebook ini menggantikan fungsi aplikasi Streamlit (alternatif demo).  
Kamu bisa meng-upload gambar, mengatur confidence threshold, dan melihat hasil deteksi secara langsung.

**Cara pakai:**
1. Jalankan semua sel di atas (kernel harus sudah terhubung).
2. Upload gambar via tombol "Pilih gambar".
3. Atur threshold confidence.
4. Klik "Deteksi" untuk melihat hasil.
5. Hasil akan muncul di bawah (gambar asli vs anotasi, ringkasan deteksi).

In [5]:
import io
import os
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from ultralytics import YOLO

# Path model (sesuaikan jika berbeda)
MODEL_PATH = os.path.join("..", "models", "best.pt")  # dari src/notebooks ke models/
model = YOLO(MODEL_PATH)
print("Model loaded.")

Model loaded.


In [6]:
def detect(image_file, conf=0.5):
    # Baca gambar dari bytes
    img_bytes = image_file.read()
    nparr = np.frombuffer(img_bytes, np.uint8)
    img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Simpan sementara
    temp_path = "temp_demo.jpg"
    cv2.imwrite(temp_path, img_bgr)
    
    # Deteksi
    results = model(temp_path, conf=conf)
    os.remove(temp_path)
    
    result = results[0]
    annotated_bgr = result.plot()
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    
    boxes = result.boxes
    return img_rgb, annotated_rgb, boxes

In [7]:
# Widget
upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Pilih gambar'
)

conf_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=1.0,
    step=0.05,
    description='Threshold:',
    continuous_update=False
)

run_button = widgets.Button(description='Deteksi')
output = widgets.Output()

# Callback tombol
def on_button_clicked(b):
    with output:
        clear_output(wait=True)
        if not upload_widget.value:
            print("⚠️ Silakan upload gambar terlebih dahulu.")
            return
        
        # upload_widget.value untuk multiple=False adalah tuple dengan satu elemen
        file_info = upload_widget.value[0]
        file_name = file_info['name']
        file_content = io.BytesIO(file_info['content'])
        
        conf = conf_slider.value
        
        print(f"Memproses {file_name} dengan confidence {conf:.2f}...")
        img_orig, img_annot, boxes = detect(file_content, conf)
        
        # Tampilkan berdampingan
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        ax1.imshow(img_orig)
        ax1.set_title("Gambar Asli")
        ax1.axis('off')
        
        ax2.imshow(img_annot)
        ax2.set_title("Hasil Deteksi")
        ax2.axis('off')
        plt.tight_layout()
        plt.show()
        
        # Ringkasan
        if boxes is not None and len(boxes.cls) > 0:
            total = len(boxes.cls)
            class_ids = boxes.cls.int().tolist()
            confs = boxes.conf.float().tolist()
            with_helmet = class_ids.count(0)
            without_helmet = class_ids.count(1)
            
            print(f"\n✅ Total pengendara terdeteksi: {total}")
            print(f"   🪖 Dengan helm: {with_helmet}")
            print(f"   ❌ Tanpa helm: {without_helmet}")
            print("\n📊 Confidence per objek:")
            for i, (cls_id, conf_val) in enumerate(zip(class_ids, confs), 1):
                cls_name = "Dengan Helm" if cls_id == 0 else "Tanpa Helm"
                print(f"   {i}. {cls_name}: {conf_val:.2f}")
        else:
            print("\n⚠️ Tidak ada objek terdeteksi. Coba turunkan confidence threshold.")

# Hubungkan tombol ke fungsi
run_button.on_click(on_button_clicked)

# Tampilkan widget
display(widgets.VBox([
    upload_widget,
    conf_slider,
    run_button,
    output
]))